In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
pip install ultralytics huggingface_hub gradio pillow

In [ ]:
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="t-Shr/Detect-AD",
    repo_type="dataset",
    allow_patterns=[
        "yolo_dataset/images/**",
        "yolo_dataset/labels/**",
        "yolo_dataset/*.yaml"
    ]
)

print(dataset_path)

In [ ]:
import os

dataset_path = "/root/.cache/huggingface/hub/datasets--t-Shr--Detect-AD/snapshots/6f4b2ee0c57fef2151eaadf4ff19adf3d756c578/yolo_dataset"

print("Dataset exists:", os.path.exists(dataset_path))
print("Train images:", len(os.listdir(dataset_path + "/images/train/toi")))

In [ ]:
import os
from pathlib import Path
import statistics

base = dataset_path

image_dir = f"{base}/images/train/toi"
label_dir = f"{base}/labels/train/toi"

stats = []

for img in sorted(os.listdir(image_dir)):
    if not img.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    label = os.path.join(
        label_dir, Path(img).stem + ".txt"
    )

    count = 0
    if os.path.exists(label):
        with open(label) as f:
            count = sum(1 for line in f if line.strip())

    stats.append((img, count))

print("Total pages:", len(stats))
print("Total annotations:", sum(x[1] for x in stats))
print("Average ads per page:", round(statistics.mean(x[1] for x in stats), 2))
print("\nAnnotations per page:\n")

for name, count in stats:
    print(f"{name}: {count}")

print("\nPages with zero annotations:",
      sum(1 for _, c in stats if c == 0))

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import os, math

zero_pages = [
    name for name, count in stats if count == 0
]

thumbs = []

for name in zero_pages:
    img = Image.open(
        os.path.join(image_dir, name)
    ).convert("RGB")

    img.thumbnail((240, 340))

    thumbs.append((name, img.copy()))

cols = 2
rows = math.ceil(len(thumbs) / cols)

sheet = Image.new(
    "RGB",
    (cols * 260, rows * 380),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, (name, img) in enumerate(thumbs):
    x = (i % cols) * 260
    y = (i // cols) * 380

    sheet.paste(img, (x, y + 30))
    draw.text((x + 5, y + 5), name, fill="black")

display(sheet)

In [ ]:
import glob
import numpy as np

areas = []

for label in glob.glob(label_dir + "/*.txt"):
    for line in open(label):
        p = line.split()
        if len(p) >= 5:
            _, cx, cy, w, h = map(float, p[:5])
            areas.append(w * h)

areas = np.array(areas)

print("Total boxes:", len(areas))
print("Below 1% page area:", np.sum(areas < 0.01))
print("1–3% page area:", np.sum((areas >= 0.01) & (areas < 0.03)))
print("3–10% page area:", np.sum((areas >= 0.03) & (areas < 0.10)))
print("Above 10% page area:", np.sum(areas >= 0.10))

print("Smallest box:", round(float(areas.min()), 5))
print("Median box area:", round(float(np.median(areas)), 5))

In [ ]:
import glob
import os

small_boxes = []

for label in glob.glob(label_dir + "/*.txt"):
    for line_no, line in enumerate(open(label), start=1):
        p = line.split()
        if len(p) >= 5:
            _, cx, cy, w, h = map(float, p[:5])
            area = w * h

            if area < 0.01:
                small_boxes.append((area, os.path.basename(label), line_no))

small_boxes.sort()

print("Small annotations:", len(small_boxes))
print("\n10 smallest annotations:")
for item in small_boxes[:10]:
    print(f"Area: {item[0]:.5f} | File: {item[1]} | Line: {item[2]}")

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

base = "/root/.cache/huggingface/hub/datasets--t-Shr--Detect-AD/snapshots/6f4b2ee0c57fef2151eaadf4ff19adf3d756c578/yolo_dataset"

image_path = f"{base}/images/train/toi/page_014.jpg"
label_path = f"{base}/labels/train/toi/page_014.txt"

img = Image.open(image_path).convert("RGB")
W, H = img.size

fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(img)

with open(label_path) as f:
    for line in f:
        cls, cx, cy, bw, bh = map(float, line.split()[:5])

        x = (cx - bw/2) * W
        y = (cy - bh/2) * H
        w = bw * W
        h = bh * H

        ax.add_patch(patches.Rectangle(
            (x, y), w, h,
            linewidth=1,
            edgecolor="red",
            facecolor="none"
        ))

ax.set_title("Page 014 — Original Advertisement Annotations")
ax.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

image_path = f"{base}/images/train/toi/page_014.jpg"
label_path = f"{base}/labels/train/toi/page_014.txt"

img = Image.open(image_path).convert("RGB")
W, H = img.size

boxes = []

with open(label_path) as f:
    for i, line in enumerate(f, start=1):
        cls, cx, cy, bw, bh = map(float, line.split()[:5])
        area = bw * bh

        if area < 0.01:
            x1 = int((cx - bw/2) * W)
            y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W)
            y2 = int((cy + bh/2) * H)
            boxes.append((area, i, x1, y1, x2, y2))

boxes.sort()

fig, axes = plt.subplots(5, 2, figsize=(14, 20))

for ax, (area, line_no, x1, y1, x2, y2) in zip(axes.flat, boxes[:10]):
    pad = 30
    crop = img.crop((
        max(0, x1-pad), max(0, y1-pad),
        min(W, x2+pad), min(H, y2+pad)
    ))
    ax.imshow(crop)
    ax.set_title(f"Label line {line_no} | Area {area:.5f}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

image_path = f"{base}/images/train/toi/page_014.jpg"
label_path = f"{base}/labels/train/toi/page_014.txt"

img = Image.open(image_path).convert("RGB")
W, H = img.size

boxes = []

with open(label_path) as f:
    for i, line in enumerate(f, start=1):
        cls, cx, cy, bw, bh = map(float, line.split()[:5])
        area = bw * bh

        if area < 0.01:
            x1 = int((cx - bw/2) * W)
            y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W)
            y2 = int((cy + bh/2) * H)
            boxes.append((area, i, x1, y1, x2, y2))

boxes.sort()

fig, axes = plt.subplots(5, 2, figsize=(14, 20))

for ax, (area, line_no, x1, y1, x2, y2) in zip(axes.flat, boxes[:10]):
    pad = 30
    crop = img.crop((
        max(0, x1-pad), max(0, y1-pad),
        min(W, x2+pad), min(H, y2+pad)
    ))
    ax.imshow(crop)
    ax.set_title(f"Label line {line_no} | Area {area:.5f}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

img = Image.open(image_path)
W, H = img.size

print("Page dimensions:", W, "x", H)
print("\nSmall advertisement boxes:\n")

with open(label_path) as f:
    for i, line in enumerate(f, start=1):
        cls, cx, cy, bw, bh = map(float, line.split()[:5])

        if bw * bh < 0.01:
            x1 = int((cx - bw/2) * W)
            y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W)
            y2 = int((cy + bh/2) * H)

            print(
                f"Line {i}: "
                f"x={x1}-{x2}, y={y1}-{y2}, "
                f"size={x2-x1}x{y2-y1}px"
            )

In [ ]:
from PIL import Image

img = Image.open(image_path)
W, H = img.size

PATCH = 2048
MODEL_SIZE = 1280

print("Original page:", W, "x", H)
print("Patch size:", PATCH)
print("Model input:", MODEL_SIZE)
print("\nSmall ad dimensions after patch resizing:\n")

with open(label_path) as f:
    count = 0
    for i, line in enumerate(f, start=1):
        cls, cx, cy, bw, bh = map(float, line.split()[:5])

        if bw * bh < 0.01:
            width = bw * W
            height = bh * H

            scaled_w = width * MODEL_SIZE / PATCH
            scaled_h = height * MODEL_SIZE / PATCH

            print(
                f"Line {i}: original={width:.0f}x{height:.0f}px "
                f"| resized={scaled_w:.0f}x{scaled_h:.0f}px"
            )
            count += 1

            if count == 10:
                break

In [ ]:
PATCH = 2048
STRIDE = 1536

with open(label_path) as f:
    boxes = []
    for line in f:
        cls, cx, cy, bw, bh = map(float, line.split()[:5])
        boxes.append((
            (cx - bw/2) * W,
            (cy - bh/2) * H,
            (cx + bw/2) * W,
            (cy + bh/2) * H
        ))

xs = list(range(0, W - PATCH + 1, STRIDE))
ys = list(range(0, H - PATCH + 1, STRIDE))

if xs[-1] != W - PATCH:
    xs.append(W - PATCH)
if ys[-1] != H - PATCH:
    ys.append(H - PATCH)

patches = [(x, y) for y in ys for x in xs]

fully_contained = 0
crossing = 0
uncovered = 0

for x1, y1, x2, y2 in boxes:
    containing = [
        (px, py) for px, py in patches
        if x1 >= px and y1 >= py
        and x2 <= px + PATCH and y2 <= py + PATCH
    ]

    if containing:
        fully_contained += 1
    else:
        crossing += 1

print("Patch grid:", len(xs), "columns x", len(ys), "rows")
print("Total patches:", len(patches))
print("Total annotations:", len(boxes))
print("Fully contained in at least one patch:", fully_contained)
print("Crossing patch boundaries / not fully contained:", crossing)

In [ ]:
for i, (x1, y1, x2, y2) in enumerate(boxes, start=1):
    if not any(
        x1 >= px and y1 >= py
        and x2 <= px + PATCH and y2 <= py + PATCH
        for px, py in patches
    ):
        print(
            f"Ad {i}: "
            f"{x2-x1:.0f} x {y2-y1:.0f}px | "
            f"position=({x1:.0f}, {y1:.0f})"
        )

In [ ]:
import glob
import os

records = []

for label in glob.glob(f"{base}/labels/train/toi/*.txt"):
    for line_no, line in enumerate(open(label), start=1):
        p = line.split()
        if len(p) < 5:
            continue

        cls, cx, cy, bw, bh = map(float, p[:5])

        records.append({
            "file": os.path.basename(label),
            "line": line_no,
            "area": bw * bh,
            "width": bw,
            "height": bh
        })

records.sort(key=lambda x: x["area"], reverse=True)

print("Largest 15 annotations:")
for r in records[:15]:
    print(
        f'{r["file"]} | line {r["line"]} | '
        f'area={r["area"]:.3f} | '
        f'w={r["width"]:.3f} | h={r["height"]:.3f}'
    )

In [ ]:
import glob
import os

outside = []
large = []

for label in glob.glob(f"{base}/labels/train/toi/*.txt"):
    for line_no, line in enumerate(open(label), start=1):
        p = line.split()
        if len(p) < 5:
            continue

        cls, cx, cy, bw, bh = map(float, p[:5])

        x1 = cx - bw/2
        y1 = cy - bh/2
        x2 = cx + bw/2
        y2 = cy + bh/2

        if x1 < 0 or y1 < 0 or x2 > 1 or y2 > 1:
            outside.append((os.path.basename(label), line_no, x1, y1, x2, y2))

        if bw * bh > 0.4:
            large.append((os.path.basename(label), line_no, bw, bh, bw*bh))

print("Boxes extending outside page:", len(outside))
print("Boxes covering >40% of page:", len(large))

for item in large:
    print(item)

In [ ]:
import glob
import os
from collections import defaultdict

page_stats = defaultdict(lambda: {
    "total": 0,
    "small": 0,
    "large": 0
})

for label in glob.glob(f"{base}/labels/train/toi/*.txt"):
    name = os.path.basename(label)

    for line in open(label):
        p = line.split()
        if len(p) < 5:
            continue

        _, cx, cy, bw, bh = map(float, p[:5])
        area = bw * bh

        page_stats[name]["total"] += 1

        if area < 0.03:
            page_stats[name]["small"] += 1
        else:
            page_stats[name]["large"] += 1

print("Pages with most small advertisements:")
for name, s in sorted(
    page_stats.items(),
    key=lambda x: x[1]["small"],
    reverse=True
)[:15]:
    print(
        f'{name}: total={s["total"]}, '
        f'small={s["small"]}, large={s["large"]}'
    )

In [ ]:

import os
import glob
import shutil
from PIL import Image

SOURCE = base
OUTPUT = "/kaggle/working/ad_patch_dataset"

PATCH = 2048
STRIDE = 1536

# Start with a clean output directory
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

os.makedirs(f"{OUTPUT}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT}/labels/train", exist_ok=True)

image_files = sorted(glob.glob(f"{SOURCE}/images/train/toi/*.jpg"))

total_patches = 0
total_labels = 0
boundary_cases = []

for image_path in image_files:
    filename = os.path.basename(image_path)
    label_path = f"{SOURCE}/labels/train/toi/{os.path.splitext(filename)[0]}.txt"

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    # Load original normalized YOLO boxes
    boxes = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                p = line.split()
                if len(p) >= 5:
                    cls, cx, cy, bw, bh = map(float, p[:5])
                    boxes.append((
                        int(cls),
                        (cx-bw/2)*W,
                        (cy-bh/2)*H,
                        (cx+bw/2)*W,
                        (cy+bh/2)*H
                    ))

    xs = list(range(0, max(1, W-PATCH+1), STRIDE))
    ys = list(range(0, max(1, H-PATCH+1), STRIDE))

    if not xs or xs[-1] != max(0, W-PATCH):
        xs.append(max(0, W-PATCH))
    if not ys or ys[-1] != max(0, H-PATCH):
        ys.append(max(0, H-PATCH))

    for py in sorted(set(ys)):
        for px in sorted(set(xs)):
            pw = min(PATCH, W-px)
            ph = min(PATCH, H-py)

            patch_boxes = []

            for cls, x1, y1, x2, y2 in boxes:
                # Keep only boxes fully inside this patch
                if (x1 >= px and y1 >= py and
                    x2 <= px+pw and y2 <= py+ph):

                    cx = ((x1+x2)/2-px)/pw
                    cy = ((y1+y2)/2-py)/ph
                    bw = (x2-x1)/pw
                    bh = (y2-y1)/ph

                    patch_boxes.append(
                        f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
                    )

            # Don't save patches with no complete ads
            if not patch_boxes:
                continue

            patch_name = (
                f"{os.path.splitext(filename)[0]}_x{px}_y{py}"
            )

            img.crop((px, py, px+pw, py+ph)).save(
                f"{OUTPUT}/images/train/{patch_name}.jpg",
                quality=95
            )

            with open(f"{OUTPUT}/labels/train/{patch_name}.txt", "w") as f:
                f.write("\n".join(patch_boxes))

            total_patches += 1
            total_labels += len(patch_boxes)

    # Record ads not fully contained in any generated patch
    for idx, (cls, x1, y1, x2, y2) in enumerate(boxes):
        contained = any(
            x1 >= px and y1 >= py and
            x2 <= px+min(PATCH, W-px) and
            y2 <= py+min(PATCH, H-py)
            for px in xs for py in ys
        )

        if not contained:
            boundary_cases.append((filename, idx+1))

print("Patch generation complete")
print("Images processed:", len(image_files))
print("Patches created:", total_patches)
print("Annotations written:", total_labels)
print("Ads without a complete patch:", len(boundary_cases))
print("Boundary examples:", boundary_cases[:20])
print("Output:", OUTPUT)


In [ ]:
from collections import Counter

counts = Counter(filename for filename, _ in boundary_cases)

print("Pages with most ads missing from complete patches:")
for filename, count in counts.most_common(15):
    print(f"{filename}: {count} ads")

In [ ]:
import os
from PIL import Image
from collections import Counter

results = []

for filename, line_no in boundary_cases:
    image_path = f"{SOURCE}/images/train/toi/{filename}"
    label_path = f"{SOURCE}/labels/train/toi/{os.path.splitext(filename)[0]}.txt"

    img = Image.open(image_path)
    W, H = img.size

    with open(label_path) as f:
        lines = f.readlines()

    cls, cx, cy, bw, bh = map(float, lines[line_no-1].split()[:5])

    x1 = (cx-bw/2)*W
    y1 = (cy-bh/2)*H
    x2 = (cx+bw/2)*W
    y2 = (cy+bh/2)*H

    area = (x2-x1)*(y2-y1)
    max_coverage = 0

    xs = list(range(0, max(1, W-PATCH+1), STRIDE))
    ys = list(range(0, max(1, H-PATCH+1), STRIDE))

    xs.append(max(0, W-PATCH))
    ys.append(max(0, H-PATCH))

    for px in set(xs):
        for py in set(ys):
            ix1 = max(x1, px)
            iy1 = max(y1, py)
            ix2 = min(x2, px+PATCH)
            iy2 = min(y2, py+PATCH)

            intersection = max(0, ix2-ix1) * max(0, iy2-iy1)
            max_coverage = max(max_coverage, intersection/area)

    results.append({
        "file": filename,
        "line": line_no,
        "width": x2-x1,
        "height": y2-y1,
        "area_pct": bw*bh*100,
        "coverage_pct": max_coverage*100
    })

print("Missing annotation categories:")

print("Large ads (>2048 px in either dimension):",
      sum(r["width"] > PATCH or r["height"] > PATCH for r in results))

print("At least 80% visible in one patch:",
      sum(r["coverage_pct"] >= 80 for r in results))

print("50–80% visible in one patch:",
      sum(50 <= r["coverage_pct"] < 80 for r in results))

print("Below 50% visible in one patch:",
      sum(r["coverage_pct"] < 50 for r in results))

print("\nFirst 20 examples:")
for r in results[:20]:
    print(
        f'{r["file"]} line {r["line"]}: '
        f'{r["width"]:.0f}x{r["height"]:.0f}px, '
        f'area={r["area_pct"]:.2f}%, '
        f'max coverage={r["coverage_pct"]:.1f}%'
    )

In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

In [ ]:
import subprocess
import sys

code = """
import torch
print("Version:", torch.__version__)
print("Path:", torch.__file__)
print("CUDA attribute:", hasattr(torch, "cuda"))

import torch.cuda
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
"""

result = subprocess.run(
    [sys.executable, "-c", code],
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(result.stdout)
print("ERROR:")
print(result.stderr)

In [ ]:

import os
from PIL import Image

OUTPUT2 = "/kaggle/working/ad_centered_patches"

os.makedirs(f"{OUTPUT2}/images", exist_ok=True)
os.makedirs(f"{OUTPUT2}/labels", exist_ok=True)

created = 0
covered_targets = 0
skipped_large = 0
saved_names = set()

for filename, line_no in boundary_cases:

    image_path = f"{SOURCE}/images/train/toi/{filename}"
    label_path = f"{SOURCE}/labels/train/toi/{os.path.splitext(filename)[0]}.txt"

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    with open(label_path) as f:
        lines = f.readlines()

    cls, cx, cy, bw, bh = map(float, lines[line_no-1].split()[:5])

    x1 = (cx-bw/2)*W
    y1 = (cy-bh/2)*H
    x2 = (cx+bw/2)*W
    y2 = (cy+bh/2)*H

    target_w = x2-x1
    target_h = y2-y1

    # Large ads remain in the original full-page dataset
    if target_w > PATCH or target_h > PATCH:
        skipped_large += 1
        continue

    # Centre patch around target, keeping it inside the page
    px = int(round((x1+x2)/2 - PATCH/2))
    py = int(round((y1+y2)/2 - PATCH/2))

    px = max(0, min(px, W-PATCH))
    py = max(0, min(py, H-PATCH))

    patch_key = (filename, px, py)

    if patch_key in saved_names:
        covered_targets += 1
        continue

    saved_names.add(patch_key)

    patch_boxes = []

    for line in lines:
        c, cx2, cy2, bw2, bh2 = map(float, line.split()[:5])

        bx1 = (cx2-bw2/2)*W
        by1 = (cy2-bh2/2)*H
        bx2 = (cx2+bw2/2)*W
        by2 = (cy2+bh2/2)*H

        # Only retain complete annotations
        if (bx1 >= px and by1 >= py and
            bx2 <= px+PATCH and by2 <= py+PATCH):

            ncx = ((bx1+bx2)/2-px)/PATCH
            ncy = ((by1+by2)/2-py)/PATCH
            nbw = (bx2-bx1)/PATCH
            nbh = (by2-by1)/PATCH

            patch_boxes.append(
                f"{int(c)} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}"
            )

    if not patch_boxes:
        continue

    patch_name = f"{os.path.splitext(filename)[0]}_center_x{px}_y{py}"

    img.crop((px, py, px+PATCH, py+PATCH)).save(
        f"{OUTPUT2}/images/{patch_name}.jpg",
        quality=95
    )

    with open(f"{OUTPUT2}/labels/{patch_name}.txt", "w") as f:
        f.write("\n".join(patch_boxes))

    created += 1
    covered_targets += 1

print("Supplemental patch generation complete")
print("New patches:", created)
print("Targets covered:", covered_targets)
print("Large ads skipped (kept in original pages):", skipped_large)
print("Output:", OUTPUT2)


In [ ]:
import os
from PIL import Image

img_dir = "/kaggle/working/ad_centered_patches/images"
lbl_dir = "/kaggle/working/ad_centered_patches/labels"

images = sorted(os.listdir(img_dir))
labels = sorted(os.listdir(lbl_dir))

print("Patch images:", len(images))
print("Label files:", len(labels))
print("Image-label count matches:", len(images) == len(labels))

total_boxes = 0
empty_labels = 0

for name in labels:
    path = os.path.join(lbl_dir, name)
    with open(path) as f:
        rows = [line.strip() for line in f if line.strip()]

    total_boxes += len(rows)
    if not rows:
        empty_labels += 1

print("Total annotations:", total_boxes)
print("Empty label files:", empty_labels)

# Check patch dimensions
sample = Image.open(os.path.join(img_dir, images[0]))
print("Sample patch dimensions:", sample.size)

In [ ]:
import os
import shutil
from pathlib import Path

SOURCE = Path(SOURCE)
GRID = Path("/kaggle/working/ad_patch_dataset")
CENTER = Path("/kaggle/working/ad_centered_patches")
OUT = Path("/kaggle/working/ad_augmented_dataset")

# Fresh output
if OUT.exists():
    shutil.rmtree(OUT)

train_images = OUT / "images" / "train"
train_labels = OUT / "labels" / "train"

train_images.mkdir(parents=True)
train_labels.mkdir(parents=True)

image_extensions = {".jpg", ".jpeg", ".png"}

def add_images(image_dir, label_dir, prefix):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    added = 0
    annotation_count = 0

    for img_path in sorted(image_dir.iterdir()):
        if img_path.suffix.lower() not in image_extensions:
            continue

        label_path = label_dir / f"{img_path.stem}.txt"

        if not label_path.exists():
            raise FileNotFoundError(f"Missing label: {label_path}")

        new_stem = f"{prefix}_{img_path.stem}"
        new_image = train_images / f"{new_stem}{img_path.suffix.lower()}"
        new_label = train_labels / f"{new_stem}.txt"

        shutil.copy2(img_path, new_image)
        shutil.copy2(label_path, new_label)

        with open(label_path) as f:
            annotation_count += sum(1 for line in f if line.strip())

        added += 1

    return added, annotation_count


# 1. Original full-page training data
full = add_images(
    SOURCE / "images/train/toi",
    SOURCE / "labels/train/toi",
    "full"
)

# 2. Fixed-grid patches
grid = add_images(
    GRID / "images",
    GRID / "labels",
    "grid"
)

# 3. Target-centred patches
center = add_images(
    CENTER / "images",
    CENTER / "labels",
    "center"
)

# Keep validation and test on original full pages
yaml_text = f"""
path: {OUT}
train: images/train
val: {SOURCE}/images/val/toi
test: {SOURCE}/images/test/toi

names:
  0: advertisement
"""

yaml_path = Path("/kaggle/working/ad_augmented.yaml")
yaml_path.write_text(yaml_text.strip() + "\n")

print("DATASET PREPARATION COMPLETE")
print("--------------------------------")
print("Full-page images:", full[0], "| annotations:", full[1])
print("Grid patches:", grid[0], "| annotations:", grid[1])
print("Centered patches:", center[0], "| annotations:", center[1])
print("--------------------------------")
print("Total training images:", full[0] + grid[0] + center[0])
print("Total training annotations:", full[1] + grid[1] + center[1])
print("YAML:", yaml_path)

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter
import hashlib

folders = {
    "grid": Path("/kaggle/working/ad_patch_dataset"),
    "centered": Path("/kaggle/working/ad_centered_patches")
}

for name, root in folders.items():
    image_dir = root / "images"
    label_dir = root / "labels"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    invalid_boxes = 0
    missing_labels = 0
    invalid_dimensions = 0
    total_boxes = 0
    box_counts = Counter()
    hashes = set()
    duplicate_images = 0

    for img_path in images:
        label_path = label_dir / f"{img_path.stem}.txt"

        if not label_path.exists():
            missing_labels += 1
            continue

        with Image.open(img_path) as img:
            if img.size != (2048, 2048):
                invalid_dimensions += 1

        digest = hashlib.md5(img_path.read_bytes()).hexdigest()
        if digest in hashes:
            duplicate_images += 1
        hashes.add(digest)

        with open(label_path) as f:
            lines = [line.strip() for line in f if line.strip()]

        box_counts[len(lines)] += 1

        for line in lines:
            values = list(map(float, line.split()))

            if len(values) != 5:
                invalid_boxes += 1
                continue

            cls, cx, cy, w, h = values

            if (
                cls != 0 or
                w <= 0 or h <= 0 or
                cx - w/2 < -0.001 or
                cy - h/2 < -0.001 or
                cx + w/2 > 1.001 or
                cy + h/2 > 1.001
            ):
                invalid_boxes += 1

            total_boxes += 1

    print(f"\n--- {name.upper()} PATCHES ---")
    print("Images:", len(images))
    print("Annotations:", total_boxes)
    print("Missing labels:", missing_labels)
    print("Invalid boxes:", invalid_boxes)
    print("Wrong dimensions:", invalid_dimensions)
    print("Exact duplicate images:", duplicate_images)
    print("Annotations per patch:", dict(sorted(box_counts.items())))

In [ ]:
from pathlib import Path

paths = [
    Path("/kaggle/working/ad_patch_dataset"),
    Path("/kaggle/working/ad_centered_patches"),
    Path("/kaggle/working/ad_augmented_dataset"),
]

for root in paths:
    print(f"\n{root}")
    print("Exists:", root.exists())

    if root.exists():
        for sub in ["images", "labels"]:
            folder = root / sub
            files = list(folder.glob("*")) if folder.exists() else []
            print(f"{sub}: {len(files)} files")
            print("Sample:", [p.name for p in files[:3]])

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/ad_patch_dataset")

for folder in [
    root / "images/train",
    root / "labels/train"
]:
    files = list(folder.rglob("*"))
    files = [p for p in files if p.is_file()]

    print("\n", folder)
    print("Files:", len(files))
    print("Samples:", [p.name for p in files[:5]])

In [ ]:
import shutil
from pathlib import Path

SOURCE = Path(SOURCE)
OUT = Path("/kaggle/working/ad_augmented_dataset")

if OUT.exists():
    shutil.rmtree(OUT)

train_images = OUT / "images/train"
train_labels = OUT / "labels/train"

train_images.mkdir(parents=True)
train_labels.mkdir(parents=True)

def add_dataset(image_dir, label_dir, prefix):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    added = 0
    boxes = 0

    for img in sorted(image_dir.rglob("*")):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        label = label_dir / f"{img.stem}.txt"

        if not label.exists():
            raise FileNotFoundError(f"Missing label: {label}")

        new_name = f"{prefix}_{img.stem}"

        shutil.copy2(img, train_images / f"{new_name}{img.suffix.lower()}")
        shutil.copy2(label, train_labels / f"{new_name}.txt")

        with open(label) as f:
            boxes += sum(1 for line in f if line.strip())

        added += 1

    return added, boxes


full = add_dataset(
    SOURCE / "images/train/toi",
    SOURCE / "labels/train/toi",
    "full"
)

# Corrected grid paths
grid = add_dataset(
    "/kaggle/working/ad_patch_dataset/images/train",
    "/kaggle/working/ad_patch_dataset/labels/train",
    "grid"
)

center = add_dataset(
    "/kaggle/working/ad_centered_patches/images",
    "/kaggle/working/ad_centered_patches/labels",
    "center"
)

yaml_text = f"""
path: {OUT}
train: images/train
val: {SOURCE}/images/val/toi
test: {SOURCE}/images/test/toi

names:
  0: advertisement
"""

Path("/kaggle/working/ad_augmented.yaml").write_text(
    yaml_text.strip() + "\n"
)

print("FULL PAGES:", full)
print("GRID PATCHES:", grid)
print("CENTERED PATCHES:", center)
print("TOTAL IMAGES:", full[0] + grid[0] + center[0])
print("TOTAL ANNOTATIONS:", full[1] + grid[1] + center[1])
print("Dataset YAML created.")

In [ ]:
from pathlib import Path
from PIL import Image

root = Path("/kaggle/working/ad_augmented_dataset")
image_dir = root / "images/train"
label_dir = root / "labels/train"

images = list(image_dir.glob("*"))
labels = list(label_dir.glob("*.txt"))

missing = []
invalid = []
empty = []
total_boxes = 0

for img in images:
    label = label_dir / f"{img.stem}.txt"

    if not label.exists():
        missing.append(img.name)
        continue

    with Image.open(img) as im:
        W, H = im.size

    rows = label.read_text().splitlines()

    if not rows:
        empty.append(img.name)

    for line_no, row in enumerate(rows, 1):
        try:
            cls, cx, cy, w, h = map(float, row.split())
            total_boxes += 1

            if (
                cls != 0 or w <= 0 or h <= 0 or
                cx - w/2 < 0 or cy - h/2 < 0 or
                cx + w/2 > 1 or cy + h/2 > 1
            ):
                invalid.append((img.name, line_no, row))

        except Exception:
            invalid.append((img.name, line_no, row))

print("Images:", len(images))
print("Labels:", len(labels))
print("Annotations:", total_boxes)
print("Missing labels:", len(missing))
print("Empty labels:", len(empty))
print("Invalid annotations:", len(invalid))

print("\nExamples of invalid annotations:", invalid[:5])

In [ ]:
!pip show torch

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO
import torch

print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/ad_augmented.yaml",
    epochs=120,
    imgsz=1280,
    batch=2,
    device=0,
    workers=2,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    patience=30,
    seed=42,
    deterministic=True,

    # Conservative augmentations for newspaper pages
    degrees=1.0,
    translate=0.03,
    scale=0.15,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.005,
    hsv_s=0.10,
    hsv_v=0.15,
    mosaic=0.10,
    close_mosaic=15,
    mixup=0.0,
    copy_paste=0.0,

    project="/kaggle/working/runs",
    name="augmented_ad_detector",
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    val=True,
    amp=True,
    verbose=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/kaggle/working/runs/augmented_ad_detector/weights/best.pt"
)

metrics = model.val(
    data="/content/ad_final.yaml",
    split="test",
    imgsz=1280,
    batch=2,
    device=0,
    plots=True,
    project="/kaggle/working/runs",
    name="augmented_test_evaluation"
)

print("\nFINAL TEST RESULTS")
print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    if "ad_final.yaml" in files or "ad_augmented.yaml" in files:
        for f in files:
            if f in ["ad_final.yaml", "ad_augmented.yaml"]:
                print(os.path.join(root, f))

In [ ]:
%%writefile /kaggle/working/ad_final.yaml
path: /root/.cache/huggingface/hub/datasets--t-Shr--Detect-AD/snapshots/6f4b2ee0c57fef2151eaadf4ff19adf3d756c578/yolo_dataset
train: images/train
val: images/val
test: images/test
names:
  0: advertisement

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/augmented_ad_detector/weights/best.pt")

metrics = model.val(
    data="/kaggle/working/ad_final.yaml",
    split="test",
    imgsz=1280,
    batch=2,
    device=0,
    plots=True
)

print("\nFINAL TEST RESULTS")
print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO(
    "/kaggle/working/runs/augmented_ad_detector/weights/best.pt"
)

test_dir = Path(
    "/root/.cache/huggingface/hub/datasets--t-Shr--Detect-AD/snapshots/6f4b2ee0c57fef2151eaadf4ff19adf3d756c578/yolo_dataset/images/test/toi"
)

image_path = next(test_dir.glob("page_008.*"), None)

print("Image:", image_path)

results = model.predict(
    source=str(image_path),
    imgsz=1280,
    conf=0.25,
    save=True,
    project="/kaggle/working/inspection",
    name="page008_augmented",
    exist_ok=True
)

print("Detected advertisements:", len(results[0].boxes))

In [ ]:
from IPython.display import display, Image
from pathlib import Path

output_dir = Path("/kaggle/working/inspection/page008_augmented")

for f in output_dir.iterdir():
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        print(f)
        display(Image(filename=str(f)))

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image
from pathlib import Path

model = YOLO(
    "/kaggle/working/runs/augmented_ad_detector/weights/best.pt"
)

IMAGE_PATH = "/kaggle/input/datasets/vanshmittal216/my-dataset/Screenshot (42).png"

results = model.predict(
    source=IMAGE_PATH,
    imgsz=1280,
    conf=0.25,
    save=True,
    project="/kaggle/working/bmw_test",
    name="prediction",
    exist_ok=True
)

print("Detected advertisements:", len(results[0].boxes))

output = Path("/kaggle/working/bmw_test/prediction")
for f in output.iterdir():
    if f.suffix.lower() in [".jpg", ".png", ".jpeg"]:
        display(Image(filename=str(f)))

In [ ]:

from ultralytics import YOLO
from PIL import Image
from pathlib import Path
import pandas as pd
import zipfile

# Model
model = YOLO(
    "/kaggle/working/runs/augmented_ad_detector/weights/best.pt"
)

# Your newspaper image path
IMAGE_PATH = "/kaggle/input/datasets/vanshmittal216/my-dataset/Screenshot (42).png"

# Output folder
OUTPUT = Path("/kaggle/working/extracted_advertisements")
OUTPUT.mkdir(parents=True, exist_ok=True)

# Open original image
image = Image.open(IMAGE_PATH).convert("RGB")
W, H = image.size

# Detect advertisements
results = model.predict(
    source=IMAGE_PATH,
    imgsz=1280,
    conf=0.25,
    iou=0.5,
    save=False,
    verbose=False
)

records = []

for i, box in enumerate(results[0].boxes):
    x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
    confidence = float(box.conf[0])

    # Small padding around the detected box
    pad_x = int((x2 - x1) * 0.01)
    pad_y = int((y2 - y1) * 0.01)

    x1 = max(0, int(x1) - pad_x)
    y1 = max(0, int(y1) - pad_y)
    x2 = min(W, int(x2) + pad_x)
    y2 = min(H, int(y2) + pad_y)

    # Crop from original resolution
    crop = image.crop((x1, y1, x2, y2))

    filename = f"advertisement_{i+1:03d}.jpg"
    crop.save(OUTPUT / filename, quality=95)

    records.append({
        "advertisement": i + 1,
        "filename": filename,
        "confidence": round(confidence, 4),
        "width": x2 - x1,
        "height": y2 - y1
    })

# Save metadata
pd.DataFrame(records).to_csv(
    OUTPUT / "advertisement_metadata.csv",
    index=False
)

# Package all crops into ZIP
zip_path = "/kaggle/working/advertisements.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT.iterdir():
        if file.is_file():
            z.write(file, arcname=file.name)

print("Original image:", W, "x", H)
print("Detected advertisements:", len(records))
print("Cropped advertisements:", len(records))
print("ZIP:", zip_path)
